In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import sys
sys.path.append('../')

In [ ]:
import pickle
import random
import h5py

from torch.utils.data import IterableDataset
from torch_geometric.data import Data, DataLoader

from Utils import AU2EV, rmsd_loss, d_mae_loss, Kabsch_alignment, pairwise_dist_to_coord, generate_fully_connected, count_negative_eig, xyz2mol, xyz2AC, visualize_mol, visualize_mol_pos, transfrom_to_rdmol, ReactionFeatureExtracter, calculate_angle_error, calculate_torsional_error, calculate_improper_error, bond_error_cnt, reaction_center_error_cnt

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import torch

In [ ]:
import torch_geometric

In [ ]:
from rdkit import Chem
from rdkit.Geometry import Point3D
from rdkit.Chem import AllChem, rdMolDescriptors
from rdkit.Chem import Descriptors

In [ ]:
from itertools import combinations

In [ ]:
REFERENCE_ENERGIES = {
    1: -13.62222753701504,
    6: -1029.4130839658328,
    7: -1484.8710358098756,
    8: -2041.8396277138045,
    9: -2712.8213146878606,
}

In [ ]:
def get_molecular_reference_energy(atomic_numbers):
    molecular_reference_energy = 0
    for atomic_number in atomic_numbers:
        molecular_reference_energy += REFERENCE_ENERGIES[atomic_number]

    return molecular_reference_energy

def generator(formula, rxn, grp):
    """ Iterates through a h5 group """

    energies = grp["wB97x_6-31G(d).energy"]
    forces = grp["wB97x_6-31G(d).forces"]
    atomic_numbers = list(grp["atomic_numbers"])
    positions = grp["positions"]
    molecular_reference_energy = get_molecular_reference_energy(atomic_numbers)

    for energy, force, positions in zip(energies, forces, positions):
        d = {
            "rxn": rxn,
            "wB97x_6-31G(d).energy": energy.__float__(),
            "wB97x_6-31G(d).atomization_energy": energy
            - molecular_reference_energy.__float__(),
            "wB97x_6-31G(d).forces": force.tolist(),
            "positions": positions,
            "formula": formula,
            "atomic_numbers": atomic_numbers,
        }

        yield d

def get_dynamics_data(formula, rxn, data):
    reactant = next(generator(formula, rxn, data[formula][rxn]["reactant"]))
    product = next(generator(formula, rxn, data[formula][rxn]["product"]))
    transition_state = next(generator(formula, rxn, data[formula][rxn]["transition_state"]))
    x = torch.tensor(reactant['atomic_numbers'], dtype=torch.long)

    reactant_pos = torch.tensor(reactant['positions'], dtype=torch.float32)
    product_pos = torch.tensor(product['positions'], dtype=torch.float32)
    transition_state_pos = torch.tensor(transition_state['positions'], dtype=torch.float32)

    energies = list()
    energies.append(torch.tensor(reactant['wB97x_6-31G(d).energy'], dtype=torch.float32))
    energies.append(torch.tensor(product['wB97x_6-31G(d).energy'], dtype=torch.float32))
    energies.append(torch.tensor(transition_state['wB97x_6-31G(d).energy'], dtype=torch.float32))
    
    return Data(x=x, reactant_pos=reactant_pos, product_pos=product_pos, transition_state_pos=transition_state_pos, energies=torch.stack(energies))

class Dataset_dynamics(IterableDataset):
    def __init__(self, hdf5_file, datasplit):
        super(Dataset_dynamics, self).__init__()
        self.hdf5_file = hdf5_file
        self.datasplit = datasplit
        assert datasplit in [
            "train",
            "valid",
            "test",
        ]
        with open('../Data/reactions_'+self.datasplit+'.pickle', 'rb') as f:
            self.datalist = pickle.load(f)

    def __iter__(self):
        with h5py.File(self.hdf5_file, "r") as f:
            data = f['data']
            i = 0
            if self.datasplit == 'train':
                random.shuffle(self.datalist)
            for formula, rxn in self.datalist:
                yield get_dynamics_data(formula, rxn, data)
                    
    def __len__(self):
        pass
    
def generate_dataloader_dynamics(hdf5_file, batch_size):
    dataloaders = {}
    dataloaders['train'] = DataLoader(dataset=Dataset_dynamics(hdf5_file, 'train'), batch_size = batch_size)
    dataloaders['val'] = DataLoader(dataset=Dataset_dynamics(hdf5_file, 'valid'), batch_size = batch_size)
    dataloaders['test'] = DataLoader(dataset=Dataset_dynamics(hdf5_file, 'test'), batch_size = batch_size)
    return dataloaders

In [ ]:
dataloader = generate_dataloader_dynamics('/home/yufeiluo/research/dataset/Transition1x/transition1x.h5', 1)

In [ ]:
feature_extractor = ReactionFeatureExtracter()

Baseline react-ot

In [ ]:
with open('res_reactot.pickle', 'rb') as f:
    temp = pickle.load(f)
pred_trans_pos_model = temp['pred_transition_state_pos']
true_trans_pos = temp['true_transition_state_pos']
reactants, products = [reactant for reactant, product in temp['reactant_product_pos']], [product for reactant, product in temp['reactant_product_pos']]
x = [torch.tensor(types) for types in temp['atom_types']]
energy_barriers = temp['true_energy_barrier_reactant']

In [ ]:
react_ot_features = {}

for i in range(len(x)):
    features = feature_extractor.extract_molecular_features(x[i], reactants[i], products[i], true_trans_pos[i], energy_barriers[i].item(), 0.0)
    reactant_bond, mol_reactant = transfrom_to_rdmol(x[i], reactants[i])
    product_bond, mol_product = transfrom_to_rdmol(x[i], products[i])
    ts_bond_pred, mol_ts_pred = transfrom_to_rdmol(x[i], pred_trans_pos_model[i])
    ts_bond_true, mol_ts_true = transfrom_to_rdmol(x[i], true_trans_pos[i])

    bond_errors = bond_error_cnt(ts_bond_pred, ts_bond_true)
    reaction_center_errors = reaction_center_error_cnt(reactant_bond, product_bond, ts_bond_true, ts_bond_pred)

    bond = ts_bond_true | reactant_bond | product_bond

    angle_errors = calculate_angle_error(true_trans_pos[i], pred_trans_pos_model[i], bond)
    torsional_errors = calculate_torsional_error(true_trans_pos[i], pred_trans_pos_model[i], bond)
    improper_errors = calculate_improper_error(true_trans_pos[i], pred_trans_pos_model[i], bond)

    react_ot_features[i] = {
        'features': features,
        'errors': {
            'bond_errors': bond_errors,
            'reaction_center_errors': reaction_center_errors,
            'angle_errors': angle_errors,
            'torsional_errors': torsional_errors,
            'improper_errors': improper_errors}
    }
    

TS-DFM

In [ ]:
with open('res_fm.pickle', 'rb') as f:
    temp = pickle.load(f)
pred_trans_pos_model = temp['fm']['pred_trans_pos']
true_trans_pos = [data.transition_state_pos for data in dataloader['test']]
reactant_pos = [data.reactant_pos for data in dataloader['test']]
product_pos = [data.product_pos for data in dataloader['test']]
x = [data.x for data in dataloader['test']]
energy_barriers = [(data.energies[-1] - data.energies[0]).item() for data in dataloader['test']]

In [ ]:
ts_dfm_features = {}

for i in range(len(x)):
    features = feature_extractor.extract_molecular_features(x[i], reactants[i], products[i], true_trans_pos[i], energy_barriers[i], 0.0)
    reactant_bond, mol_reactant = transfrom_to_rdmol(x[i], reactants[i])
    product_bond, mol_product = transfrom_to_rdmol(x[i], products[i])
    ts_bond_pred, mol_ts_pred = transfrom_to_rdmol(x[i], pred_trans_pos_model[i])
    ts_bond_true, mol_ts_true = transfrom_to_rdmol(x[i], true_trans_pos[i])

    bond_errors = bond_error_cnt(ts_bond_pred, ts_bond_true)
    reaction_center_errors = reaction_center_error_cnt(reactant_bond, product_bond, ts_bond_true, ts_bond_pred)

    bond = ts_bond_true | reactant_bond | product_bond

    angle_errors = calculate_angle_error(true_trans_pos[i], pred_trans_pos_model[i], bond)
    torsional_errors = calculate_torsional_error(true_trans_pos[i], pred_trans_pos_model[i], bond)
    improper_errors = calculate_improper_error(true_trans_pos[i], pred_trans_pos_model[i], bond)

    ts_dfm_features[i] = {
        'features': features,
        'errors': {
            'bond_errors': bond_errors,
            'reaction_center_errors': reaction_center_errors,
            'angle_errors': angle_errors,
            'torsional_errors': torsional_errors,
            'improper_errors': improper_errors}
    }

In [ ]:
with open('react_ot_vs_dfm_features.pickle', 'wb') as f:
    pickle.dump({'react_ot_features': react_ot_features, 'ts_dfm_features': ts_dfm_features}, f)